In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, to_json
import pandas as pd
import json

In [0]:
BASE_PATH = "/Volumes/workspace/project_data_football_raw/pontuacao_raw"

In [0]:
# Função para obter última rodada processada
def get_ultima_rodada_processada():
    try:
        result = spark.sql("""
        SELECT COALESCE(MAX(rodada), 0) as ultima
        FROM project_data_football_bronze.pontuacao_rodada
        """).collect()[0]["ultima"]

        return result
    except:
        return 0

In [0]:
# Obtém a última rodada processada da tabela bronze
ultima_processada = get_ultima_rodada_processada()

print("Última rodada processada:", ultima_processada)

# Lista os diretórios das rodadas no caminho base
rodadas_dirs = dbutils.fs.ls(BASE_PATH)

In [0]:
for r in rodadas_dirs:

    # Ignora diretórios que não seguem o padrão "rodada="
    if not r.name.startswith("rodada="):
        continue

    # Extrai o número da rodada do nome do diretório
    rodada = int(r.name.replace("rodada=", "").replace("/", ""))

    # Pula rodadas já processadas
    if rodada <= ultima_processada:
        continue

    print(f"🔄 Processando rodada {rodada}")

    # Lista arquivos da rodada e ordena do mais recente para o mais antigo
    arquivos = dbutils.fs.ls(r.path)
    arquivos = sorted(arquivos, key=lambda x: x.name, reverse=True)

    # Se não houver arquivos, pula para a próxima rodada
    if not arquivos:
        continue

    # Seleciona o arquivo mais recente
    ultimo_arquivo = arquivos[0].path
    print(f"Arquivo: {ultimo_arquivo}")

    # Lê o arquivo JSON como DataFrame Spark
    df_raw = spark.read.json(ultimo_arquivo)

    # Converte o DataFrame para dict Python (simulando API)
    data = df_raw.toPandas().iloc[0].to_dict()

    # Obtém os dados dos atletas
    atletas = data.get("atletas", {})

    # Se não houver atletas, pula para a próxima rodada
    if not atletas:
        print(f"⚠️ Rodada {rodada} sem atletas")
        continue

    lista_atletas = []

    # Monta lista de dicts com dados dos atletas
    for atleta_id, dados in atletas.items():
        dados["atleta_id"] = int(atleta_id)
        lista_atletas.append(dados)

    df_pontuacao = spark.createDataFrame(pd.DataFrame(lista_atletas))

    df_pontuacao = df_pontuacao \
        .withColumn("rodada", lit(rodada)) \
        .withColumn("dt_ingestao", current_timestamp()) \
        .withColumn("scout", to_json(col("scout")))

    tabela_pontos = "project_data_football_bronze.pontuacao_rodada"

    if spark.catalog.tableExists(tabela_pontos):
        dt_pontos = DeltaTable.forName(spark, tabela_pontos)
        
        # Chave composta: atleta_id E rodada
        dt_pontos.alias("target") \
            .merge(
                df_pontuacao.alias("source"),
                "target.atleta_id = source.atleta_id AND target.rodada = source.rodada"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
    else:
        # Salva os dados no Delta Lake, permitindo evolução de schema
        df_pontuacao.write \
            .format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true")\
            .saveAsTable("project_data_football_bronze.pontuacao_rodada")

        print(f"✅ Rodada {rodada} carregada")